# NPO+KL: WarBias training

Run the cells in order to prepare the data, see one optimizer step on CPU, and configure the six models. Enable training separately.

NPO suppresses target completions using their token-mean log-likelihood ratio against a fixed reference. Reference KL preserves predictions on retain data.

## Setup

Install `requirements-unlearning.txt` and configure the checkpoint and output paths with the [setup guide](SETUP.md). The data and CPU cells do not load model weights.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

repo = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "src/fairforget/unlearning/objectives.py").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open this notebook from a fair_forget checkout.")
sys.path.insert(0, str(repo / "src"))
from fairforget.unlearning import notebook_matrix, objectives, warbias_data  # noqa: E402

print("Algorithm module:", objectives.__name__)
print("PyTorch:", torch.__version__)

METHOD = "npo_kl"
RUN_TRAINING = False

Algorithm module: fairforget.unlearning.objectives
PyTorch: 2.9.1


## WarBias data

Use the current 1,007 aligned triplets in Ukrainian and English. Train on Ukrainian text. Keep English for paired evaluation.

Split whole claim components. Protect every component used by a QA case from training and development. Use seeded SHA256 order and round 20% of the remaining components up to six development components. The default split contains 220 training, 70 development and 717 held-out triplets. The same item IDs and splits apply to both languages.

Use complete S sentences as forget targets and complete C sentences as correction targets, with an empty prompt. Use complete U sentences for in-domain reference KL. This measures preservation of WarBias text; general capability needs separate evidence.


In [2]:
settings_path = repo / "configs/unlearning/local.json"
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
data_value = (
    os.environ.get("FAIRFORGET_DATA_ROOT")
    or settings.get("data_roots", {}).get("warbias_final")
    or repo / "data/unlearning/warbias_final"
)
DATA_ROOT = Path(data_value).expanduser().resolve()
if not DATA_ROOT.exists():
    warbias_data.prepare(repo, DATA_ROOT)
BINDING = warbias_data.validate_bundle(DATA_ROOT, repo)
print(json.dumps(warbias_data.overview(repo), ensure_ascii=False, indent=2))
print("Training language:", BINDING["training_language"])
print("Evaluation languages:", ", ".join(BINDING["evaluation_languages"]))
print("Validated WarBias data binding.")

{
  "protocol": "warbias_final_v1",
  "triplets_per_language": 1007,
  "qa_per_language": 240,
  "training_language": "uk",
  "evaluation_languages": [
    "uk",
    "en"
  ],
  "source": {
    "repository": "FairForget/WarBias",
    "triplet_revision": "9363a1d1b6936ad3d37e52f9ce1e426d2a0573b0",
    "git_source_commit": "36c49eb65ec370bb7e792dcb4d9b5e63f8bf0609",
    "files": {
      "triplets_uk": {
        "path": "warbias-artifacts/data/uk.jsonl",
        "sha256": "61692dee0036584e7b27014a7e509f751dd372e046903c394c3cd686a1069cdb",
        "hf_filename": "data/uk/unassigned.jsonl"
      },
      "triplets_en": {
        "path": "warbias-artifacts/data/en.jsonl",
        "sha256": "6611322c7102161a8170bb8e6811b0244e0d6c1a8b5234f6dee0328ffaebc218",
        "hf_filename": "data/en/unassigned.jsonl"
      },
      "qa_uk": {
        "path": "warbias-artifacts/data/qa_uk.jsonl",
        "sha256": "f7c85372d89c1ced05ff4419bef67f1012fe6214d8d3e089bb3917542dfc620b"
      },
      "qa_en": 

## CPU example

Synthetic tensors illustrate one optimizer step. Training uses the WarBias rows and settings loaded below.

In [3]:
# Synthetic tensors for one optimizer step on CPU.
torch.manual_seed(42)
forget_logits = torch.nn.Parameter(torch.randn(2, 4, 7))
retain_logits = torch.nn.Parameter(torch.randn(2, 4, 7))
reference_forget = forget_logits.detach().clone()
reference_retain = retain_logits.detach().clone() + 0.1 * torch.randn(2, 4, 7)
labels = torch.tensor([[1, 2, 3, 4], [2, 3, 4, 5]])
completion_mask = torch.tensor([[False, True, True, True], [False, True, True, False]])
retain_mask = torch.tensor([[True, True, True, True], [True, True, True, False]])


def completion_logps(logits):
    return logits.log_softmax(-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)


parameters = [forget_logits, retain_logits]
optimizer = torch.optim.AdamW(parameters, lr=0.01)
optimizer.zero_grad(set_to_none=True)

In [4]:
loss = objectives.npo_kl_loss(
    completion_logps(forget_logits),
    completion_logps(reference_forget),
    completion_mask,
    retain_logits,
    reference_retain,
    retain_mask,
    beta=0.1,
    retain_weight=1.0,
    retain_reduction="token_mean",
)
loss.total.backward()
assert torch.all(forget_logits.grad[~completion_mask] == 0)
assert reference_forget.grad is None and reference_retain.grad is None
torch.nn.utils.clip_grad_norm_(parameters, 1.0)
optimizer.step()
print("NPO+KL CPU optimizer step completed.")

NPO+KL CPU optimizer step completed.


## Training

Enable `RUN_TRAINING` to train all six models in sequence. Each model has a separate log and output directory. Each run uses one GPU with enough memory for the model. All three methods use the same WarBias training and development rows.

In [5]:
MODELS_ROOT = os.environ.get("FAIRFORGET_MODELS_ROOT")
output_value = os.environ.get("FAIRFORGET_OUTPUT") or str(
    Path(settings.get("output_root", repo / "outputs/unlearning")) / METHOD
)
OUTPUT = Path(output_value).expanduser().resolve()
PLAN = notebook_matrix.method_plan(
    repo,
    METHOD,
    protocol="warbias_final",
    data_root=DATA_ROOT,
    models_root=MODELS_ROOT,
)
assert len(PLAN) == 6, "Expected six models in the training plan."
for item in PLAN:
    recipe = item["recipe"]
    print(
        item["model_alias"],
        recipe["model_id"],
        recipe["model_revision"],
        "GPUs:",
        item["required_gpus"],
    )
print("Models:", len(PLAN), "| Method:", METHOD, "| Dataset: WarBias")

qwen3_8b Qwen/Qwen3-8B b968826d9c46dd6066d109eabc6255188de91218 GPUs: 1
llama31_8b_instruct meta-llama/Llama-3.1-8B-Instruct 0e9e39f249a16976918f6564b8830bc894c89659 GPUs: 1
gemma3_12b_it google/gemma-3-12b-it 96b6f1eccf38110c56df3a15bffe176da04bfd80 GPUs: 1
lapa12b_instruct lapa-llm/lapa-v0.1.2-instruct 2969fd997f07b33727c6f10795d78ef23cc5c037 GPUs: 1
mamay12b_v2 INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v2.0 3cb02013b6b37cfe10f4b3da0fcddc1acef7f244 GPUs: 1
mamay27b_v2 INSAIT-Institute/MamayLM-Gemma-3-27B-IT-v2.0 ad1bdfed9646939358adfcba9c88bb4031b50ed7 GPUs: 1
Models: 6 | Method: npo_kl | Dataset: WarBias


In [6]:
if RUN_TRAINING:
    from importlib.metadata import version

    required_versions = {
        "torch": "2.9.1",
        "transformers": "4.57.6",
        "peft": "0.17.1",
        "accelerate": "1.15.0",
    }
    for package, expected in required_versions.items():
        if version(package).split("+")[0] != expected:
            raise RuntimeError(
                f"Use requirements-unlearning.txt: {package}=={expected} is required for full training."
            )
    if not torch.cuda.is_available() or torch.cuda.device_count() < max(
        item["required_gpus"] for item in PLAN
    ):
        raise RuntimeError("Allocate one GPU with enough memory for the selected model.")
    # Validate all checkpoints and data before starting the first model.
    recipe_paths = notebook_matrix.prepare_recipes(PLAN, OUTPUT)
    environment = dict(os.environ)
    environment["PYTHONPATH"] = str(repo / "src") + os.pathsep + environment.get("PYTHONPATH", "")
    environment.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    execution = {"method": METHOD, "models": {}}
    execution_path = OUTPUT / "matrix_execution.json"
    for item, recipe_path in zip(PLAN, recipe_paths, strict=True):
        alias = item["model_alias"]
        command = [
            sys.executable,
            "-m",
            "fairforget.unlearning.trainer",
            "--config",
            str(recipe_path),
            "--output",
            str(OUTPUT / alias),
        ]
        record = {"status": "running", "recipe": str(recipe_path), "command": command}
        execution["models"][alias] = record
        execution_path.write_text(json.dumps(execution, indent=2) + "\n")
        print("Training", alias, flush=True)
        # A new process releases GPU memory between models.
        with (OUTPUT / (alias + ".log")).open("w") as log:
            completed = subprocess.run(
                command,
                cwd=repo,
                env=environment,
                stdout=log,
                stderr=subprocess.STDOUT,
                check=False,
            )
        record.update(
            status="completed" if completed.returncode == 0 else "failed",
            returncode=completed.returncode,
        )
        execution_path.write_text(json.dumps(execution, indent=2) + "\n")
        completed.check_returncode()
    assert len(execution["models"]) == len(PLAN) == 6
    print("All six model runs completed.")
else:
    print("All six models are configured. Training is disabled.")

All six models are configured. Training is disabled.


## Evaluation and implementation

Evaluate the same held-out WarBias IDs in Ukrainian and English before and after training. A score over the full release is a descriptive diagnostic because it includes training and development rows. All metrics, including severity slices, come from Fair-UK.

The [baseline notebook](warbias_baseline.ipynb) provides the current per-checkpoint metrics. Final before/after analysis needs compatible paired bindings and current-text neutral controls. See the [WarBias workflow](WARBIAS_UNLEARNING.md).

- [Data binding](../src/fairforget/unlearning/warbias_data.py)
- [Loss implementations](../src/fairforget/unlearning/objectives.py)
- [Trainer](../src/fairforget/unlearning/trainer.py)

Each run saves a checkpoint, loss history and token-exposure log.
